# Build a Minimal Droid — an autonomous software engineering agent

[Factory](https://factory.ai) builds **Droids**: autonomous software engineering
agents. You hand a Droid a ticket; it explores the codebase, forms a plan,
edits files, runs the test suite, iterates until things pass, and reports back
with a structured record of what it did.

Production Droids are large systems — sandboxed execution, repo-scale code
search, git integration, review gates, fleet orchestration. But the **core
loop** that makes a Droid a Droid is surprisingly small:

```
ticket ──▶ ┌─────────────────────────────────┐
           │  agent loop                     │
           │  explore → reproduce → fix →    │ ──▶ structured run report
           │  verify → (repeat until green)  │      (goal, actions, mistakes,
           └────┬───────────────┬────────────┘       final status)
                │ tools         │
                ▼               ▼
        read / write files   run shell commands
        (jailed to a repo)   (tests, scripts)
```

In this notebook we rebuild that loop in roughly 150 lines using `vidbyte-sdk`:

1. **A demo repo with a real bug** — so the droid has something to fix.
2. **Four tools** — `list_files`, `read_file`, `write_file`, `run` — the droid's hands.
3. **The agent** — a system prompt encoding the Droid operating procedure.
4. **A continual trace** — `TraceOption.continual(ActionTrace)` gives us the
   structured "what happened" artifact that real Droids produce, for free.

> ⚠️ The `run` tool executes shell commands on your machine, jailed to a scratch
> workspace. That's fine for this demo; production systems run this in a sandbox.

## Setup

In [ ]:
%pip install -q vidbyte-sdk python-dotenv requests

import os
from dotenv import load_dotenv

load_dotenv()             # .env next to this notebook
load_dotenv("../../.env") # or at the repo root

PROVIDER = os.getenv("VIDBYTE_COOKBOOK_PROVIDER", "openai")
MODEL = os.getenv("VIDBYTE_COOKBOOK_MODEL", "gpt-4.1")

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), \
    "Set OPENAI_API_KEY (or ANTHROPIC_API_KEY + provider overrides) in .env first."

## Step 1 — Create the demo repo

A tiny inventory module with a classic bug: `apply_discount` treats a 0–100
percentage as if it were a 0–1 fraction, so a 10% discount on \$100 returns
**-900.0** instead of 90.0. The test suite catches it; the droid's job is to
make the suite green.

In [ ]:
import shutil
from pathlib import Path

WORKSPACE = Path("workspace").resolve()
shutil.rmtree(WORKSPACE, ignore_errors=True)
WORKSPACE.mkdir()

DEMO_FILES = {
    "inventory.py": """def apply_discount(price, percent):
    \"\"\"Apply a percentage discount (0-100) to a price.\"\"\"
    return price - price * percent


def cart_total(cart, discount_percent=0):
    \"\"\"Total a cart of {'price': float, 'qty': int} items, then discount.\"\"\"
    subtotal = sum(item["price"] * item["qty"] for item in cart)
    return round(apply_discount(subtotal, discount_percent), 2)
""",
    "test_inventory.py": """import unittest

from inventory import apply_discount, cart_total


class TestInventory(unittest.TestCase):
    def test_ten_percent_discount(self):
        self.assertEqual(apply_discount(100.0, 10), 90.0)

    def test_no_discount(self):
        self.assertEqual(apply_discount(50.0, 0), 50.0)

    def test_cart_total_with_discount(self):
        cart = [{"price": 25.0, "qty": 2}, {"price": 50.0, "qty": 1}]
        self.assertEqual(cart_total(cart, 10), 90.0)


if __name__ == "__main__":
    unittest.main()
""",
}

for name, content in DEMO_FILES.items():
    (WORKSPACE / name).write_text(content, encoding="utf-8")

# Snapshot the originals so we can diff the droid's edits later.
ORIGINALS = {name: content for name, content in DEMO_FILES.items()}
print("Demo repo created at", WORKSPACE)

## Step 2 — Give the droid hands: four tools

Plain Python functions become tools via `@tool` — the SDK derives the calling
schema from type hints and docstrings. Two safety properties worth noticing,
because every production coding agent has industrial versions of them:

- **Path jail** — file tools resolve every path inside the workspace and
  refuse anything that escapes it.
- **Bounded execution** — `run` has a timeout and truncates output, so a hung
  or chatty command can't wedge the loop or flood the context window.

In [ ]:
import subprocess

from vidbyte import tool


def _safe_path(rel_path: str) -> Path:
    p = (WORKSPACE / rel_path).resolve()
    p.relative_to(WORKSPACE)  # raises ValueError if the path escapes the jail
    return p


@tool
def list_files() -> list[str]:
    """List every file in the repository, as paths relative to the repo root."""
    return sorted(str(p.relative_to(WORKSPACE)) for p in WORKSPACE.rglob("*") if p.is_file())


@tool
def read_file(path: str) -> str:
    """Read one file from the repository. path is relative to the repo root."""
    return _safe_path(path).read_text(encoding="utf-8")


@tool
def write_file(path: str, content: str) -> str:
    """Overwrite one file in the repository with new content. Creates the file
    (and parent folders) if it does not exist. Returns a short confirmation."""
    target = _safe_path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
    return f"wrote {len(content)} chars to {path}"


@tool
def run(command: str) -> dict:
    """Run a shell command from the repo root (e.g. 'python -m unittest -v').
    Returns exit_code, stdout, and stderr (each truncated to 4000 chars)."""
    proc = subprocess.run(
        command, shell=True, cwd=WORKSPACE, capture_output=True, text=True, timeout=60
    )
    return {
        "exit_code": proc.returncode,
        "stdout": proc.stdout[-4000:],
        "stderr": proc.stderr[-4000:],
    }

## Step 3 — The droid itself

The system prompt encodes the Droid operating procedure: **reproduce before
you fix, verify before you claim**. That discipline — never trusting an edit
until the suite proves it — is most of what separates an agent that *seems*
to fix bugs from one that does.

The second constructor argument worth a close look is
`trace_option=TraceOption.continual(ActionTrace)`. While the droid works, a
separate trace agent watches a read-only snapshot of the run every few
iterations and maintains a structured artifact — goal, actions taken,
mistakes, current status. That's the minimal version of the session report a
production Droid hands back with its PR, and we get it without touching the
main loop.

In [ ]:
from vidbyte import BaseAgent, TraceOption
from vidbyte.trace.continual import ActionTrace

DROID_PROMPT = """You are a software engineering droid. You are given a ticket
and a repository you can inspect and modify with your tools.

Operating procedure — follow it strictly:
1. EXPLORE: list the files and read the ones relevant to the ticket.
2. REPRODUCE: run the test suite (python -m unittest -v) and confirm you can
   see the failure described in the ticket. Never fix what you haven't reproduced.
3. FIX: make the smallest correct change. Preserve existing style and public
   interfaces. Do not refactor beyond the ticket's scope.
4. VERIFY: re-run the full test suite. If anything fails, keep iterating.
5. REPORT: when the suite is green, summarize the root cause, the exact change
   you made, and the verification result.

Never claim success without a passing test run in this session."""

droid = BaseAgent(
    name="droid",
    system_prompt=DROID_PROMPT,
    provider=PROVIDER,
    model_name=MODEL,
    tools=[list_files, read_file, write_file, run],
    max_iterations=30,
    trace_option=TraceOption.continual(ActionTrace, every_n_iterations=3),
)

## Step 4 — Hand it a ticket

This is the whole interface: a ticket goes in, an autonomous run happens,
a report comes out.

In [ ]:
TICKET = """TICKET-1042: Discounts produce absurd totals

Customer reports: a $100 cart with a 10% discount shows a total of -900.0.
Expected: 90.0. The discount percent is passed on a 0-100 scale.

Acceptance criteria: the full test suite (python -m unittest) passes."""

reply = droid.run(TICKET)
print(reply.content)

## Step 5 — Trust, but verify

The droid says it's done. We don't take its word for it — we run the suite
ourselves and diff every file against the pre-run snapshot.

In [ ]:
import difflib

check = subprocess.run(
    "python -m unittest -v", shell=True, cwd=WORKSPACE, capture_output=True, text=True
)
print("Independent test run exit code:", check.returncode)
print(check.stderr[-1500:])

for name, before in ORIGINALS.items():
    after = (WORKSPACE / name).read_text(encoding="utf-8")
    if after != before:
        diff = difflib.unified_diff(
            before.splitlines(keepends=True), after.splitlines(keepends=True),
            fromfile=f"a/{name}", tofile=f"b/{name}",
        )
        print("".join(diff))

## Step 6 — The structured run report

`reply.metadata["trace"]` (also `droid.last_trace`) holds the artifact the
trace agent maintained during the run. This is the handoff document: another
agent — or a human reviewer — can pick up the work from here without replaying
the transcript.

In [ ]:
import json

trace = reply.metadata.get("trace") or droid.last_trace
print(json.dumps(trace, indent=2))

## What production Droids add

The skeleton you just built scales into the real thing along a few axes:

- **Sandboxing** — real droids execute in isolated containers/VMs, not your laptop.
- **Repo-scale navigation** — code search and dependency graphs instead of `list_files` over four files (see the SDK's built-in `GrepTool`).
- **Git-native workflow** — branch, commit, open a PR, respond to review comments.
- **Review gates and policies** — what the droid may touch, and what needs human sign-off.
- **Fleet orchestration** — many droids on many tickets, with shared memory of the codebase.

Things to try from here: give the droid a multi-file feature ticket instead of
a bug; add a `git_commit` tool; raise `every_n_iterations` and compare trace
quality; or swap `max_iterations` down to 5 and watch how it prioritizes.